# 分类 + 聚类通用建模模板（Python 3.11 · 美赛 MCM/ICM 直接复用）

> **一句话用法**：把附件数据放进 `导入数据/`，改下面第 2 节的 `CONFIG`（数据路径、标签列），
> 然后 `Run All`。中间所有清洗、预处理、聚类、分类、画图、导出都自动完成。

## 这个 Notebook 解决什么问题

美赛拿到的数据往往是「脏」的：Excel/CSV 带 BOM、数字列混着空格和文本、类别列有前后空格、
`999` / `未填` 之类的哨兵值、缺测列、重复行……如果每次都重写一遍清洗代码，既慢又容易出错，
论文里也讲不清「可复现性」。本模板把这些步骤固化成**带 fit/transform 的复用对象**：

| 模块 | 作用 | 可复用点 |
| --- | --- | --- |
| `load_table` | 读 csv/xlsx，统一编码、列名、缺失标记、数值列、日期列 | 换任何附件只需改路径 |
| `data_audit` | 输出缺失率 / 异常值 / 基数 / 重复行体检表 | 论文「数据预处理」一节直接引用 |
| `TabularPreprocessor` | 填补 → 缩尾 → 标准化 → 独热编码，**记住全部训练参数** | 换新数据 `transform` 不会数据泄漏 |
| `evaluate_k` / `find_elbow` | 肘部法 + 轮廓系数 + CH + DB 自动选簇数 | 避免「拍脑袋定 K」 |
| `run_kmeans` / 层次 / DBSCAN | 三类聚类 + 簇画像 + 可视化 + 一致性对比 | 多方法互证，评委喜欢 |
| `run_classification` | 4 个分类器 × 分层交叉验证 + 混淆矩阵 + AUC | 有标签列时自动执行 |
| `run_linkage` | 把簇标签当新特征，检验「先分群再判别」是否有效 | 分类 × 聚类联动，常用创新点 |
| `export_results` / `predict_new` | 结果写 `输出数据/`，新数据一键预测 | 交付 / 灵敏度分析 |

## 方法简介（论文可直接引用的公式）

- **K-Means**：最小化簇内平方和 $SSE=\sum_{k=1}^{K}\sum_{i\in C_k}\|x_i-\mu_k\|^2$，
  用 `n_init=10` 多次初始化取最优；K-Means++ 初始化降低陷入局部最优的概率。
- **轮廓系数**：$s(i)=\dfrac{b(i)-a(i)}{\max\{a(i),b(i)\}}\in[-1,1]$，越大表示簇内越紧、簇间越远。
- **Calinski-Harabasz**（越大越好）与 **Davies-Bouldin**（越小越好）作为选 K 的旁证。
- **Ward 层次聚类**：每步合并使簇内方差增量最小的两簇，谱系图给出聚类树状结构；
  与 K-Means 用 **ARI（调整兰德指数）** 比较结果稳定性。
- **DBSCAN**：基于密度，$(eps, min\_samples)$ 决定簇，能自动识别离群点（标签 `-1`），
  适合做异常样本筛查；`eps` 由 k-距离曲线的拐点自动给出。
- **有监督分类**：逻辑回归 / 随机森林 / SVM / KNN，分层 $k$ 折交叉验证，
  指标用 Accuracy、Macro-Precision/Recall/F1、ROC-AUC； Macro 平均避免类别不平衡掩盖问题。
- **前提**：聚类前必须标准化，否则量纲大的变量会主导欧氏距离；缺失率过高的列直接剔除。

## 运行环境

Python **3.11**（已在 3.11.9 实测）；依赖：

```bash
pip install numpy pandas scipy scikit-learn matplotlib openpyxl
```

无标签列时把 `CONFIG.target = None`，Notebook 会**自动跳过全部分类环节**，只做聚类。

In [2]:
# ============================================================
# 0. 环境准备：版本检查 + 全局设置
# 全部代码兼容 Python 3.11；若在别的内核运行，这里会直接给出提示
# ============================================================
import sys
import warnings
import platform
from pathlib import Path
from dataclasses import dataclass, field, fields

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# scipy：层次聚类的谱系图 / 距离矩阵
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

# sklearn：聚类、降维、评估、分类、交叉验证
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from pandas.api.types import is_numeric_dtype, is_datetime64_any_dtype

# ---- 版本检查：避免 3.8/3.9 的语法差异（例如 str | None 类型注解）----
if sys.version_info[:2] < (3, 11):
    print(f"[警告] 当前 Python {platform.python_version()}，本模板按 3.11 编写，建议切换内核。")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ---- 画图全局设置：中文 + 负号 + 清晰度 ----
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "SimSun", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False          # 负号不显示为方块
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# ---- pandas 显示设置：中文列对齐、少用科学计数法 ----
pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

print("Python      :", platform.python_version())
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("matplotlib  :", matplotlib.__version__)
import sklearn
print("scikit-learn:", sklearn.__version__)

Python      : 3.11.9
numpy       : 2.4.6
pandas      : 3.1.0
matplotlib  : 3.11.2
scikit-learn: 1.9.1


### 1. 换一份数据，到底要改哪里？（照着做就行）

#### ① 数据该长什么样

**一行 = 一个样本，一列 = 一个指标**。数值列（如 pm25）或文字类别列（如地区）都可以，
列名叫中文也没问题。例如这个模板自带的数据：

| date | region | pm25 | temperature | traffic_index | fuel_type | survey_score |
| --- | --- | --- | --- | --- | --- | --- |
| 2024-01-01 | 东部 | 61.7 | 15.9 | 117.0 | 电力 | 72.4 |
| 2024-01-02 | 东部 | 69.9 | 12.3 | 109.5 | 生物质 | 60.3 |

你**不需要**提前做任何事：BOM、前后空格、`999` 哨兵、被存成文本的数字、重复行、
缺失值、量纲差异、文字类别转独热编码——模板全自动处理。

唯一要判断的是：**这份数据有没有「标签列」**？

- 有（如 `风险等级`、`地区`、`是否超标`）→ 填 `target="列名"`，聚类 + 分类 + 联动全跑；
- 没有 → `target=None`，**只做聚类**，第 10、11 节自动跳过。

#### ② 三步走

1. 把新文件放进 `导入数据/` 文件夹（`.csv` / `.xlsx` / `.xls` 都支持）；
2. 改第 2 节 `CONFIG` 的 **3 个值**：`data_path`（文件名）、`target`（标签列或 `None`）、`tag`（本次输出的名字）；
3. `Run All`，结果自动写进 `输出数据/分类聚类_{tag}_*`。**中间所有函数一行都不用改。**

更省事：跳过 `CONFIG`，直接跳到第 13 节用一行 `run_on(...)`（下面 ③ 有例子）。

#### ③ 三个照抄场景（把文件名换成你自己的）

```python
# 场景 A：换一个 CSV，有标签列「风险等级」→ 聚类 + 分类 + 联动
bundle = run_on("问题E数据.csv", target="风险等级", tag="problem_e")

# 场景 B：换一个 Excel，没有标签 → 只做聚类；顺手指定工作表
bundle = run_on("附件2.xlsx", target=None, tag="cluster_only", sheet_name="Sheet1")

# 场景 C：换数据 + 调参数（最多试 10 个簇、不生成图片、换标准化方式）
bundle = run_on("附件2.xlsx", target=None, tag="cluster_only",
                k_max=10, make_plots=False, scale="minmax")
```

> `run_on` 的第一个参数**只写文件名**即可，它会自动去 `导入数据/` 里找；
> 也可以写完整绝对路径（如 `r"D:\某文件夹\数据.xlsx"`）。
> 换一份数据就换一个 `tag`，否则会覆盖上一份的结果。

#### ④ 拿到结果之后

```python
bundle["clu_df"]        # 原始表 + cluster / dbscan_cluster / hier_cluster 列（每行属于哪一簇）
bundle["profile"]       # 簇画像：各簇的数值均值、类别众数、样本数 → 论文里给簇命名用
bundle["k_table"]       # K=2..k_max 的四项选 K 指标
bundle["cls"]["report"] # 有标签时：4 个分类器的交叉验证对比
bundle["df"]            # 清洗后的数据；bundle["X"] 是标准化后的特征矩阵
# 新来一批数据，直接复用训练好的模型（不用重新拟合）：
result = predict_new(bundle, 新数据_df)
```
表格和图片同时会写到 `输出数据/`，Excel 是多 Sheet 汇总，可直接粘进论文。

#### ⑤ 哪些参数要改 / 不用改

| 参数 | 什么时候改 | 默认 |
| --- | --- | --- |
| `data_path` | **必改**：换新数据文件 | 自带示例 csv |
| `target` | **必改**：标签列名；没有就 `None` | `"region"` |
| `tag` | **建议改**：换题目就换名字，避免覆盖 | `"A_原始数据"` |
| `id_cols` | 数据里有编号/日期等不该进模型的列时加进去 | `("date",)` |
| `features` | 只想用某几列建模时显式列出，如 `["pm25","temperature"]` | `None`（自动挑） |
| `na_markers` | 题目用别的数表示缺测（如 `-1`）；或 `999` 其实是真实值要删掉 | 999/-999/未填… |
| `k_max` / `outlier` / `scale` | 做灵敏度分析时 | 8 / IQR 缩尾 / z-score |
| `make_plots` / `RUN_EXAMPLES` | 不想生成图、不想跑末尾演示 | `True` / `True` |

**自动推断会做这些事**（不用你管）：剔除标签列、`id_cols`、日期列、缺失率 > 50% 的列、
常数列、唯一值 > 30 的文本列；数值列填补 → 缩尾 → 标准化，文字列众数填补 → 独热编码。

#### ⑥ 常见坑

| 现象 | 原因 / 解决 |
| --- | --- |
| `KeyError: '风险等级'` | 标签列名写错了。先跑 `df.columns` 看真实列名，注意列名会被自动去掉首尾空格 |
| 标签是连续分数（如 0~100 得分） | 模板自动按三分位分箱成 `低/中/高` 再分类 |
| `FileNotFoundError` | 文件不在 `导入数据/`；或写成了 `导入数据/xxx.csv`（用 `run_on` 时只写文件名） |
| 结果被上一份覆盖 | `tag` 没换；同名是覆盖，换新 `tag` 才会新增一组 |
| 数据里 `999` 是真实值 | 把 `999` 从 `Config.na_markers` 里删掉 |
| 编辑器里改了没生效 | Notebook 不会自动重载磁盘变更，**重新打开文件**再运行 |

### 输出文件与内存：再运行一次会发生什么？

- 图片统一写到 `输出数据/分类聚类_{tag}_{图名}.png`，**文件名固定、同名覆盖**。
  只要 `tag` 不变，再跑 10 次也不会多出一个文件、磁盘不会越滚越大；
  **换了 `tag` 才会新增一整组文件**（旧版演示曾叫 `raw_demo`，所以出现了重复的一组）。
- 每个 cell 的输出是**替换**而不是追加，重新 `Run All` 不会把旧图叠进 Notebook。
- `save_fig` 在保存后会 `plt.close(fig)` 释放画布，反复运行不会在内核里堆积 figure
  （早期版本没关画布，这是唯一会真正「占内存」的地方，现已修复）。
- 想完全不产生图片：`Config(make_plots=False)`；
  想跳过末尾两个演示示例、只跑一次全流程：把下面的 `RUN_EXAMPLES` 改成 `False`。
- 想清掉某一组历史产物：调用 `clean_outputs("raw_demo")`（只删本模板生成的文件）。

In [4]:
# ============================================================
# 2. 项目路径 + 配置区（本 Notebook 唯一需要用户修改的地方）
# ============================================================
def find_project_root(marker="导入数据"):
    """从当前工作目录向上找含有 `导入数据` 文件夹的目录。

    Jupyter 的工作目录可能是项目根目录，也可能是 聚类模型/ 子目录，
    两种情况下都能正确定位数据目录。找不到就退回当前目录（换题目时不影响使用）。
    """
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / marker).is_dir():
            return base
    return Path.cwd()


ROOT = find_project_root()
DATA_DIR = ROOT / "导入数据" if (ROOT / "导入数据").is_dir() else ROOT
OUT_DIR = ROOT / "输出数据" if (ROOT / "输出数据").is_dir() else ROOT
OUT_DIR.mkdir(parents=True, exist_ok=True)   # 不存在则自动创建，保证导出不报错
print("项目根目录 :", ROOT)
print("数据目录   :", DATA_DIR)
print("输出目录   :", OUT_DIR)


@dataclass
class Config:
    """一键配置：所有建模开关集中在这里，方便论文里做灵敏度分析。"""
    # ---------- 数据 ----------
    data_path: object = None            # 数据文件路径；None = 自动在 DATA_DIR 里找第一个 csv/xlsx
    sheet_name: object = 0             # Excel 第几个工作表（名字或序号）
    id_cols: tuple = ("date",)         # 不参与建模的标识列（编号、日期等），仍会保留在输出表里
    target: object = "region"          # 有监督分类的标签列；None = 只做聚类、跳过分类
    features: object = None            # 参与建模的特征列；None = 自动推断（排除 id/日期/标签/缺测列）
    tag: str = "demo"                  # 输出文件名前缀，换数据集时改它避免覆盖
    make_plots: bool = True            # 是否生成并保存图片；False = 只算表不画图（省时省内存）

    # ---------- 读入与清洗 ----------
    na_markers: tuple = (999, 999.0, -999, "未填", "NA", "N/A", "null", "未知")
    # ↑ 哨兵值：题目常用 999 / -999 表示缺测。若数据里 999 是真实值，请从这里删掉。
    drop_duplicates: bool = True       # 删除完全重复的行
    drop_high_missing: bool = True     # 删除缺失率过高的列
    missing_threshold: float = 0.50    # 缺失率 > 50% 的列直接不参与建模（可保留在原始输出表中）

    # ---------- 预处理 ----------
    impute_num: str = "median"         # 数值列填补：'median' 中位数 / 'mean' 均值 / 'zero' 填 0
    impute_cat: str = "mode"           # 类别列填补：'mode' 众数 / 'unknown' 填 '未知'
    add_missing_indicator: bool = True # 为发生过缺失的数值列增加 xxx_missing 指示列（保留缺测信息）
    outlier: str = "winsor_iqr"        # 异常值：'winsor_iqr' 1.5IQR 缩尾 / 'winsor_3sigma' / 'none'
    scale: str = "standard"            # 标准化：'standard' z-score / 'minmax' / 'robust' / 'none'
    max_onehot_cardinality: int = 30   # 类别列唯一值超过这个数视为编号/文本，自动排除
    onehot_drop_first: bool = False    # 独热编码是否丢掉第一列（线性模型可设 True 避免共线）

    # ---------- 聚类 ----------
    k_min: int = 2                     # 尝试的最小簇数
    k_max: int = 8                     # 尝试的最大簇数
    dbscan_min_samples: int = 5        # DBSCAN 的 min_samples（一般取 2×特征数，可自行调）
    dbscan_eps_quantile: float = 0.90  # 若拐点失效，用 k-距离的该分位数作为 eps 兜底

    # ---------- 分类 ----------
    max_label_classes: int = 20        # 数值标签唯一值超过它 → 自动按三分位分箱成 低/中/高
    n_splits: int = 5                  # 分层交叉验证折数
    test_size: float = 0.25            # 留出法测试集比例
    random_state: int = 42             # 全局随机种子：保证结果可复现（论文必须写）


# 示例 1 默认配置：直接吃「原始脏数据」，并以 region 作为演示用标签列
CONFIG = Config(data_path=DATA_DIR / "mcm_final_raw.csv", target="region", tag="A_原始数据")

# 末尾第 14～16 节是「演示示例」，会各自再跑一遍全流程并产出一组图片。
# 正式复用时把它改成 False，就只跑第 3～12 节这一套，输出文件少一半以上。
RUN_EXAMPLES = True
print("配置：", CONFIG)
print("是否运行末尾演示示例：", RUN_EXAMPLES)

项目根目录 : D:\git项目
数据目录   : D:\git项目\导入数据
输出目录   : D:\git项目\输出数据
配置： Config(data_path=WindowsPath('D:/git项目/导入数据/mcm_final_raw.csv'), sheet_name=0, id_cols=('date',), target='region', features=None, tag='A_原始数据', make_plots=True, na_markers=(999, 999.0, -999, '未填', 'NA', 'N/A', 'null', '未知'), drop_duplicates=True, drop_high_missing=True, missing_threshold=0.5, impute_num='median', impute_cat='mode', add_missing_indicator=True, outlier='winsor_iqr', scale='standard', max_onehot_cardinality=30, onehot_drop_first=False, k_min=2, k_max=8, dbscan_min_samples=5, dbscan_eps_quantile=0.9, max_label_classes=20, n_splits=5, test_size=0.25, random_state=42)
是否运行末尾演示示例： True


### 3. 通用数据读取 `load_table`

自动处理美赛附件里最常见的坑：

- `.csv` 依次尝试 `utf-8-sig`（去掉 Excel BOM）/ `utf-8` / `gbk`；`.xlsx` 用 `openpyxl`；
- 列名、字符串值统一 `strip()`，避免 `" 南部 "` 和 `"南部"` 被当成两个类别；
- `999`、`未填`、空串等哨兵统一变成 `NaN`；
- 数字被存成文本（例如 `" 58.25 "`）时自动 `to_numeric` 转回数值；
- 名字含 `date/time/日期` 且能解析的列转成 `datetime`（后续自动排除出特征矩阵）；
- 删除空行、全空列、重复行，并打印每一步改了多少，方便写进论文的预处理说明。

In [6]:
# ============================================================
# 3. 数据读取与初清洗
# ============================================================
# 通用的「文字型缺失」标记；数值型哨兵放 CONFIG.na_markers，由用户按题目决定
TEXT_NA = ["", " ", "  ", "-", "?", "NA", "N/A", "na", "n/a", "null", "NULL",
           "None", "nan", "NaN", "未填", "未知", "缺失", "无"]


def resolve_data_path(cfg):
    """确定数据文件：显式给了就用，没给就在数据目录里自动挑一个表格文件。"""
    if cfg.data_path is not None:
        p = Path(cfg.data_path)
        if not p.is_absolute():                       # 相对路径按项目根目录解析
            p = ROOT / p
        if not p.exists():
            raise FileNotFoundError(f"找不到数据文件：{p}\n请检查 CONFIG.data_path。")
        return p
    candidates = sorted([p for p in DATA_DIR.iterdir()
                         if p.suffix.lower() in (".csv", ".xlsx", ".xls")])
    if not candidates:
        raise FileNotFoundError(f"{DATA_DIR} 里没有 csv/xlsx 文件。")
    print(f"[提示] 未指定 data_path，自动选用：{candidates[0].name}")
    return candidates[0]


def load_table(cfg):
    """读入表格 → 返回 (清洗后的 DataFrame, 元信息 dict)。

    元信息里记录读入了多少行、删了多少重复行、哪些列被转成数值 / 日期，
    这些数字可以直接写进论文的「数据清洗」段落。
    """
    path = resolve_data_path(cfg)
    suffix = path.suffix.lower()
    # 只读入「文字型」缺失标记；数值哨兵（999 等）留到第 3.5 步统一替换，
    # 这样才能在报告里准确统计「有多少个 999 被判为缺测」，便于论文说明处理依据。
    na_values = list(dict.fromkeys(TEXT_NA))

    # ---------- 3.1 按格式读取，CSV 自动尝试多种编码 ----------
    if suffix in (".xlsx", ".xls"):
        df = pd.read_excel(path, sheet_name=cfg.sheet_name, na_values=na_values)
        enc_used = "Excel(openpyxl)"
    else:
        df, enc_used = None, None
        for enc in ("utf-8-sig", "utf-8", "gbk", "gb18030"):
            try:
                df = pd.read_csv(path, encoding=enc, na_values=na_values,
                                 skipinitialspace=True)
                enc_used = enc
                break
            except UnicodeDecodeError:
                continue
        if df is None:
            raise UnicodeDecodeError("all", b"", 0, 1, "常见编码都无法解码该 CSV，请检查文件编码。")

    n_raw = len(df)
    # ---------- 3.2 列名规范化：去空格、去 BOM、重名列加后缀 ----------
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    seen, new_cols = {}, []
    for c in df.columns:
        seen[c] = seen.get(c, 0) + 1
        new_cols.append(c if seen[c] == 1 else f"{c}_{seen[c]}")
    df.columns = new_cols

    # 删掉整行全空 / 整列全空
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all").copy()

    # ---------- 3.3 非数值列：去首尾空格 → 再把文字型缺失标记置为 NaN ----------
    for c in df.columns:
        if not is_numeric_dtype(df[c]) and not is_datetime64_any_dtype(df[c]):
            df[c] = df[c].map(lambda v: v.strip() if isinstance(v, str) else v)
            df[c] = df[c].replace({m: np.nan for m in TEXT_NA})

    # ---------- 3.4 数字被存成文本：能转成数值的比例 ≥ 90% 就转回来 ----------
    numeric_turned, date_cols = [], []
    for c in df.columns:
        if is_numeric_dtype(df[c]) or is_datetime64_any_dtype(df[c]):
            continue
        s = df[c]
        valid = s.notna().sum()
        if valid == 0:
            continue
        conv = pd.to_numeric(s, errors="coerce")
        if conv.notna().sum() >= 0.90 * valid:       # 文本数字列（如 ' 58.25 '）
            df[c] = conv.astype(float)
            numeric_turned.append(c)
        else:
            # 日期列：列名含 date/time/日期/时间，且绝大多数能解析成日期
            hint = any(k in c.lower() for k in ("date", "time", "日期", "时间"))
            if hint:
                dt = pd.to_datetime(s, errors="coerce")
                if dt.notna().sum() >= 0.90 * valid:
                    df[c] = dt
                    date_cols.append(c)

    # ---------- 3.5 数值哨兵（999 / -999 等）：转成 NaN，并记录替换个数 ----------
    sentinel_hits = {}
    numeric_sentinels = [v for v in cfg.na_markers if isinstance(v, (int, float))]
    for c in df.select_dtypes(include="number").columns:
        hit = df[c].isin(numeric_sentinels).sum()
        if hit:
            sentinel_hits[c] = int(hit)
            df[c] = df[c].replace({v: np.nan for v in numeric_sentinels})

    # ---------- 3.6 重复行 ----------
    n_dup = int(df.duplicated().sum())
    if cfg.drop_duplicates and n_dup:
        df = df.drop_duplicates().reset_index(drop=True)

    meta = dict(path=path, encoding=enc_used, n_raw=n_raw, n_final=len(df),
                n_duplicates=n_dup, numeric_turned=numeric_turned,
                date_cols=date_cols, sentinel_hits=sentinel_hits)
    print(f"读取成功 : {path.name}  （编码 {enc_used}）")
    print(f"数据规模 : {n_raw} 行 → {len(df)} 行，{df.shape[1]} 列；删除重复行 {n_dup} 行")
    print(f"文本转数值列 : {numeric_turned or '无'}；日期列：{date_cols or '无'}")
    print(f"哨兵值转缺失 : {sentinel_hits or '无'}")
    return df, meta


# 用它读一次，看看脏数据被整理成了什么样
df, META = load_table(CONFIG)
print("\n前 5 行：")
print(df.head().to_string())
print("\n字段类型：")
print(df.dtypes.to_string())

读取成功 : mcm_final_raw.csv  （编码 utf-8-sig）
数据规模 : 381 行 → 366 行，8 列；删除重复行 15 行
文本转数值列 : ['pm25']；日期列：['date']
哨兵值转缺失 : {'traffic_index': 6}

前 5 行：
        date region    pm25  temperature  traffic_index fuel_type  survey_score  household_income
0 2024-02-26   西部 32.1551      24.6503       121.2980    天然气       77.0827               NaN
1 2024-02-08   北部 58.2526      22.6777        98.8713    生物质       67.5126               NaN
2 2024-02-10   南部 38.0487          NaN            NaN    生物质       60.7735               NaN
3 2024-01-03   北部 60.6144      14.9992       107.2503      电力       61.0147               NaN
4 2024-02-05   南部 56.0553      21.5517        88.0174      电力       61.2994        5,602.0670

字段类型：
date                datetime64[us]
region                         str
pm25                       float64
temperature                float64
traffic_index              float64
fuel_type                      str
survey_score               float64
household_income           float64


### 4. 数据体检 `data_audit`

建模之前先「体检」：每列的缺失率、唯一值个数、极值、IQR 异常点数量、是否常数列。
输出的 `audit` 表可以直接贴进论文附录，也是决定「哪些列能进模型」的依据。

In [8]:
# ============================================================
# 4. 数据体检：缺失 / 异常 / 类型 / 重复，一张表说清楚
# ============================================================
def data_audit(df, cfg, plot=True):
    """输出逐列体检表；plot=True 时额外画缺失率条形图与数值箱线图。"""
    rows = []
    for c in df.columns:
        s = df[c]
        miss = int(s.isna().sum())
        rec = dict(列名=c, 类型=str(s.dtype), 缺失数=miss,
                   缺失率=round(miss / len(df), 4), 唯一值=int(s.nunique(dropna=True)))
        if is_numeric_dtype(s):
            q1, q3 = s.quantile(0.25), s.quantile(0.75)
            iqr = q3 - q1
            rec.update(最小值=round(float(s.min()), 3) if s.notna().any() else np.nan,
                       中位数=round(float(s.median()), 3) if s.notna().any() else np.nan,
                       最大值=round(float(s.max()), 3) if s.notna().any() else np.nan,
                       IQR异常点=int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()),
                       建议="剔除(缺测过多)" if (cfg.drop_high_missing and
                                             miss / len(df) > cfg.missing_threshold)
                       else ("常数列,剔除" if s.nunique(dropna=True) <= 1 else "保留"))
        else:
            rec.update(最小值="", 中位数="", 最大值="", IQR异常点=0,
                       建议=("日期/标识列,不建模" if is_datetime64_any_dtype(s)
                             else "独热编码" if s.nunique(dropna=True) <= cfg.max_onehot_cardinality
                             else "高基数,剔除"))
        rows.append(rec)
    audit = pd.DataFrame(rows)
    print("===== 数据体检表 =====")
    print(audit.to_string(index=False))
    print(f"\n重复行：{int(df.duplicated().sum())}；总样本：{len(df)}")

    if plot and cfg.make_plots:      # make_plots=False 时连画布都不创建
        # 图 4-1：缺失率（只画有缺失的列，没有缺失就跳过）
        has_miss = audit[audit["缺失率"] > 0]
        if len(has_miss):
            fig, ax = plt.subplots(figsize=(7, max(2.6, 0.5 * len(has_miss) + 1)))
            ax.barh(has_miss["列名"], has_miss["缺失率"] * 100, color="#d95f5f")
            ax.axvline(cfg.missing_threshold * 100, ls="--", color="#444",
                       label=f"剔除阈值 {cfg.missing_threshold:.0%}")
            ax.set_xlabel("缺失率 (%)"); ax.set_title("各列缺失率")
            ax.legend(); fig.tight_layout()
            save_fig(fig, "缺失率", cfg)

        # 图 4-2：数值列箱线图（各列先标准化，仅用于同一张图里比较形状）
        num_cols = [c for c in df.columns if is_numeric_dtype(df[c])]
        if num_cols:
            z = df[num_cols].apply(lambda s: (s - s.mean()) / (s.std() or 1))
            fig, ax = plt.subplots(figsize=(max(6, 1.1 * len(num_cols)), 4))
            ax.boxplot([z[c].dropna() for c in num_cols])   # 不用 labels 参数，兼容 matplotlib 3.9+
            ax.set_xticks(range(1, len(num_cols) + 1))
            ax.set_xticklabels(num_cols, rotation=25, ha="right")
            ax.set_ylabel("标准化后的取值"); ax.set_title("数值列分布与异常点")
            fig.tight_layout()
            save_fig(fig, "箱线图", cfg)
    return audit


def save_fig(fig, name, cfg):
    """统一保存图片：同名覆盖 + 用完关闭画布。

    两个关键点：
    1) 文件名 = 分类聚类_{tag}_{图名}.png，固定不变 → 反复运行只覆盖，不新增文件；
    2) plt.close(fig) 释放 figure 对象 → 反复 Run All 不会在内核里堆积画布占内存。
    """
    if not getattr(cfg, "make_plots", True):
        plt.close(fig)               # 不画图也要关掉，避免空画布堆积
        return None
    path = OUT_DIR / f"分类聚类_{cfg.tag}_{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()                       # Jupyter 里正常显示（inline 后端会先渲染成输出）
    plt.close(fig)                   # 显示完成后立刻释放内存
    print(f"[图片] 已保存：{path.name}（同名覆盖）")
    return path


audit = data_audit(df, CONFIG)

===== 数据体检表 =====
            列名           类型  缺失数  缺失率  唯一值     最小值     中位数     最大值  IQR异常点               建议
            date datetime64[us]       0  0.0000      90                                           0 日期/标识列,不建模
          region            str       0  0.0000       4                                           0           独热编码
            pm25        float64      18  0.0492     344    18.6900    50.4260    93.2070          1               保留
     temperature        float64       5  0.0137     356    11.8600    23.2560    30.6550          0               保留
   traffic_index        float64      15  0.0410     351    43.7890    98.9720   150.0030          4               保留
       fuel_type            str       8  0.0219       4                                           0           独热编码
    survey_score        float64      18  0.0492     342    44.0990    69.7360    99.6440          3               保留
household_income        float64     318  0.8689      42 3,115.6080 5,240.4430 9,9

### 5. 可复用预处理器 `TabularPreprocessor`

这是整个模板的核心。它把「任意表格 → sklearn 能直接吃的数值矩阵」的过程封装成
`fit` / `transform` 两步：

- **fit 阶段**只在训练集上统计：中位数/众数、缩尾边界、均值/标准差/分位数、类别清单；
- **transform 阶段**对新数据使用**同一套参数**，不会把测试集信息泄漏进训练；
- 自动剔除：标识列、日期列、标签列、缺失率过高的列、常数列、高基数文本列；
- 数值列：填补 → 1.5×IQR 缩尾（不删样本，保住样本量）→ 标准化；
- 类别列：众数填补 → 独热编码；新数据里出现训练时没见过的类别，全部编码为 0；
- 发生缺失的数值列额外生成 `xxx_missing` 指示变量，保留「缺测本身也是信息」。

> 聚类必须用标准化后的矩阵：欧氏距离对量纲敏感，不标准化时 `收入(几千)` 会完全压倒 `评分(几十)`。

In [10]:
# ============================================================
# 5. 表格预处理器：fit 记住参数，transform 复用到任意新数据
# ============================================================
class TabularPreprocessor:
    """把 DataFrame 变成数值特征矩阵 X（列名可读），并支持对将来数据做同样变换。

    属性（fit 之后可用）
    --------------------
    numeric_cols_ / categorical_cols_ : 实际参与建模的列
    feature_names_                   : 输出矩阵的列名（顺序与 X 一致）
    dropped_cols_                    : 被剔除的列及原因（写论文预处理依据时用）
    fill_num_ / fill_cat_ / bounds_ / center_ / scale_ / categories_ : 全部训练参数
    """

    def __init__(self, cfg):
        self.cfg = cfg
        self.numeric_cols_, self.categorical_cols_ = [], []
        self.indicator_cols_, self.dropped_cols_ = [], {}
        self.fill_num_, self.fill_cat_, self.bounds_ = {}, {}, {}
        self.center_, self.scale_ = {}, {}
        self.categories_, self.feature_names_ = {}, []

    # ---------- 5.1 挑选可用特征 ----------
    def _pick_columns(self, df):
        excluded = set(self.cfg.id_cols or ())
        if self.cfg.target is not None:
            excluded.add(self.cfg.target)
        for c in df.columns:
            # 日期列不直接进模型；即使没被解析成 datetime，名字带日期的列也排除
            if is_datetime64_any_dtype(df[c]) or any(
                    k in str(c).lower() for k in ("date", "time", "日期", "时间")):
                excluded.add(c)
        cols = [c for c in df.columns if c not in excluded]
        if self.cfg.features is not None:            # 用户显式指定时以其为准
            cols = [c for c in self.cfg.features if c in df.columns]

        for c in cols:
            s = df[c]
            miss_rate = s.isna().mean()
            if miss_rate == 1:
                self.dropped_cols_[c] = "全空列"; continue
            if self.cfg.drop_high_missing and miss_rate > self.cfg.missing_threshold:
                self.dropped_cols_[c] = f"缺失率 {miss_rate:.1%} > {self.cfg.missing_threshold:.0%}"
                continue
            nunique = s.nunique(dropna=True)
            if nunique <= 1:
                self.dropped_cols_[c] = "常数列（无区分度）"; continue
            if is_numeric_dtype(s):
                self.numeric_cols_.append(c)
            elif nunique > self.cfg.max_onehot_cardinality:
                self.dropped_cols_[c] = f"高基数文本列（{nunique} 个取值）"
            else:
                self.categorical_cols_.append(c)
        if not self.numeric_cols_ and not self.categorical_cols_:
            raise ValueError("没有任何可用特征，请检查 CONFIG.features / missing_threshold。")

    # ---------- 5.2 fit：只在训练数据上统计变换参数 ----------
    def fit(self, df):
        self._pick_columns(df)
        cfg = self.cfg
        for c in self.numeric_cols_:
            s = pd.to_numeric(df[c], errors="coerce").astype(float)
            if cfg.impute_num == "mean":
                fill = float(s.mean())
            elif cfg.impute_num == "zero":
                fill = 0.0
            else:
                fill = float(s.median())
            if not np.isfinite(fill):                # 整列异常时的兜底
                fill = 0.0
            self.fill_num_[c] = fill
            # 若原表已自带同名指示列（例如上游清洗流程生成的 xxx_missing），
            # 就不要再生成一份，否则输出矩阵会出现重复列名。
            if (s.isna().any() and cfg.add_missing_indicator
                    and f"{c}_missing" not in df.columns):
                self.indicator_cols_.append(c)

            filled = s.fillna(fill)
            if cfg.outlier == "winsor_iqr":
                q1, q3 = filled.quantile(0.25), filled.quantile(0.75)
                iqr = q3 - q1
                lo, hi = float(q1 - 1.5 * iqr), float(q3 + 1.5 * iqr)
            elif cfg.outlier == "winsor_3sigma":
                mu, sd = float(filled.mean()), float(filled.std())
                lo, hi = mu - 3 * sd, mu + 3 * sd
            else:
                lo, hi = -np.inf, np.inf
            self.bounds_[c] = (lo, hi)
            clipped = filled.clip(lo, hi)

            if cfg.scale == "minmax":
                self.center_[c], self.scale_[c] = float(clipped.min()), float(clipped.max() - clipped.min())
            elif cfg.scale == "robust":
                q1, q3 = clipped.quantile(0.25), clipped.quantile(0.75)
                self.center_[c], self.scale_[c] = float(clipped.median()), float(q3 - q1)
            elif cfg.scale == "standard":
                self.center_[c], self.scale_[c] = float(clipped.mean()), float(clipped.std())
            else:
                self.center_[c], self.scale_[c] = 0.0, 1.0
            if not np.isfinite(self.scale_[c]) or self.scale_[c] == 0:
                self.scale_[c] = 1.0                # 防止常数列除零

        for c in self.categorical_cols_:
            s = df[c]
            if cfg.impute_cat == "unknown":
                self.fill_cat_[c] = "未知"
            else:
                mode = s.mode(dropna=True)
                self.fill_cat_[c] = str(mode.iat[0]) if len(mode) else "未知"
            cats = sorted({str(v).strip() for v in s.dropna().unique()})
            if cfg.onehot_drop_first and len(cats) > 1:
                cats = cats[1:]                   # 丢掉第一类作参照组
            self.categories_[c] = cats

        # 输出列名与 transform 的顺序严格一致
        names = list(self.numeric_cols_) + [f"{c}_missing" for c in self.indicator_cols_]
        for c in self.categorical_cols_:
            names += [f"{c}={v}" for v in self.categories_[c]]
        self.feature_names_ = self._unique(names)
        return self

    @staticmethod
    def _unique(names):
        """列名去重兜底：万一原表列名与编码列名撞车，自动加 _2/_3 后缀。"""
        seen, out = {}, []
        for n in names:
            seen[n] = seen.get(n, 0) + 1
            out.append(n if seen[n] == 1 else f"{n}_{seen[n]}")
        return out

    # ---------- 5.3 transform：用 fit 记住的参数处理任意新数据 ----------
    def transform(self, df):
        missing_cols = [c for c in (self.numeric_cols_ + self.categorical_cols_) if c not in df.columns]
        if missing_cols:
            raise KeyError(f"新数据缺少这些建模所需列：{missing_cols}")
        blocks = []
        for c in self.numeric_cols_:
            s = pd.to_numeric(df[c], errors="coerce").astype(float).fillna(self.fill_num_[c])
            lo, hi = self.bounds_[c]
            z = (s.clip(lo, hi) - self.center_[c]) / self.scale_[c]
            blocks.append(z.to_numpy(dtype=float))
        for c in self.indicator_cols_:
            blocks.append(df[c].isna().to_numpy(dtype=float))
        for c in self.categorical_cols_:
            s = df[c].map(lambda v: str(v).strip() if pd.notna(v) else self.fill_cat_[c])
            for v in self.categories_[c]:
                blocks.append((s == v).to_numpy(dtype=float))
        if len(blocks) != len(self.feature_names_):
            raise RuntimeError("transform 与 fit 的特征数不一致，请检查预处理器实现。")
        # 列名统一取 fit 时保存的版本，保证训练 / 推理两侧特征名严格一致
        X = pd.DataFrame(np.column_stack(blocks), columns=self.feature_names_, index=df.index)
        return X

    def fit_transform(self, df):
        return self.fit(df).transform(df)


# 在当前数据上试跑一次
pre = TabularPreprocessor(CONFIG).fit(df)
X = pre.transform(df)
print("参与建模的数值列 :", pre.numeric_cols_)
print("参与建模的类别列 :", pre.categorical_cols_)
print("被剔除的列及原因 :", pre.dropped_cols_ or "无")
print(f"特征矩阵 X 形状   : {X.shape}（样本 × 特征）")
print("输出特征名       :", pre.feature_names_)
print("\nX 前 3 行：")
print(X.head(3).to_string())

参与建模的数值列 : ['pm25', 'temperature', 'traffic_index', 'survey_score']
参与建模的类别列 : ['fuel_type']
被剔除的列及原因 : {'household_income': '缺失率 86.9% > 50%'}
特征矩阵 X 形状   : (366, 12)（样本 × 特征）
输出特征名       : ['pm25', 'temperature', 'traffic_index', 'survey_score', 'pm25_missing', 'temperature_missing', 'traffic_index_missing', 'survey_score_missing', 'fuel_type=天然气', 'fuel_type=煤', 'fuel_type=生物质', 'fuel_type=电力']

X 前 3 行：
     pm25  temperature  traffic_index  survey_score  pm25_missing  temperature_missing  traffic_index_missing  survey_score_missing  fuel_type=天然气  fuel_type=煤  fuel_type=生物质  fuel_type=电力
0 -1.3215       0.5228         1.5750        0.7512        0.0000               0.0000                 0.0000                0.0000            1.0000        0.0000            0.0000          0.0000
1  0.5507       0.0443        -0.0392       -0.3273        0.0000               0.0000                 0.0000                0.0000            0.0000        0.0000            1.0000          0.0000
2 -0

### 6. 自动确定簇数 K

不拍脑袋定 $K$：对 $K=2,\dots,k_{max}$ 全部拟合 K-Means，同时算 4 个指标：

| 指标 | 方向 | 说明 |
| --- | --- | --- |
| SSE（惯性） | 越小越好，看**拐点** | 拐点用「归一化曲线到首尾连线距离最大」自动定位 |
| 轮廓系数 | 越大越好 | 直接衡量簇内紧密度与簇间分离度 |
| Calinski-Harabasz | 越大越好 | 簇间/簇内方差比 |
| Davies-Bouldin | 越小越好 | 簇内散度与簇间距离之比 |

默认**以轮廓系数最大的 K 为主**，并把肘部法给出的 K 一并打印；若两者不一致，
论文里可写「综合轮廓系数与肘部法，并考虑簇的可解释性，取 $K=\dots$」。

In [12]:
# ============================================================
# 6. 选 K：肘部法 + 轮廓系数 + CH + DB（通用拐点函数 find_elbow 后面 DBSCAN 也会用）
# ============================================================
def find_elbow(values):
    """通用拐点检测：把曲线归一化后，找距离「首尾连线」最远的点。

    无需额外依赖（kneed 库），返回拐点在数组中的下标；曲线太短时返回中点。
    """
    y = np.asarray(values, dtype=float)
    n = len(y)
    if n < 3:
        return n // 2
    rng = y.max() - y.min()
    yn = (y - y.min()) / (rng if rng > 0 else 1)     # 归一化到 [0,1]
    xn = np.linspace(0, 1, n)
    # 点到首尾连线的垂直距离（带符号取绝对值）
    line = yn[0] + (yn[-1] - yn[0]) * xn
    dist = np.abs(yn - line) / np.sqrt(1 + (yn[-1] - yn[0]) ** 2)
    return int(np.argmax(dist))


def evaluate_k(X, cfg):
    """逐个 K 拟合 K-Means，返回指标表与推荐簇数 (k_table, k_best, k_elbow)。"""
    k_list = [k for k in range(cfg.k_min, cfg.k_max + 1) if k < len(X)]
    rows = []
    for k in k_list:
        km = KMeans(n_clusters=k, n_init=10, random_state=cfg.random_state).fit(X)
        lab = km.labels_
        rows.append(dict(K=k,
                         SSE=km.inertia_,
                         轮廓系数=silhouette_score(X, lab),
                         CH=calinski_harabasz_score(X, lab),
                         DB=davies_bouldin_score(X, lab)))
        print(f"  K={k}: SSE={km.inertia_:,.1f}  轮廓={rows[-1]['轮廓系数']:.4f}  "
              f"CH={rows[-1]['CH']:.1f}  DB={rows[-1]['DB']:.4f}")
    k_table = pd.DataFrame(rows)
    k_best = int(k_table.loc[k_table["轮廓系数"].idxmax(), "K"])
    k_elbow = int(k_table["K"].iloc[find_elbow(k_table["SSE"].to_numpy())])
    print(f"\n轮廓系数最优 K = {k_best}；肘部法 K = {k_elbow}"
          f"（两者不一致时以轮廓系数为主，并结合簇的业务含义）")
    return k_table, k_best, k_elbow


def plot_k_diagnostics(k_table, k_best, k_elbow, cfg):
    if not cfg.make_plots:
        return
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    items = [("SSE", "SSE（越小越好，看拐点）", "#3b7dd8"),
             ("轮廓系数", "轮廓系数（越大越好）", "#d95f5f"),
             ("CH", "Calinski-Harabasz（越大越好）", "#4a9d6a"),
             ("DB", "Davies-Bouldin（越小越好）", "#a06ac9")]
    for ax, (col, title, color) in zip(axes.ravel(), items):
        ax.plot(k_table["K"], k_table[col], "o-", color=color, lw=2)
        ax.axvline(k_best, ls="--", color="#d95f5f", label=f"轮廓最优 K={k_best}")
        ax.axvline(k_elbow, ls=":", color="#666", label=f"肘部 K={k_elbow}")
        ax.set_xlabel("簇数 K"); ax.set_title(title); ax.legend(fontsize=8)
    fig.suptitle("簇数选择：四项内部评价指标", fontsize=13)
    fig.tight_layout()
    save_fig(fig, "选K指标", cfg)


k_table, K_BEST, K_ELBOW = evaluate_k(X, CONFIG)
plot_k_diagnostics(k_table, K_BEST, K_ELBOW, CONFIG)
print("\n指标汇总表：")
print(k_table.to_string(index=False))

  K=2: SSE=1,493.3  轮廓=0.1590  CH=67.9  DB=2.1826
  K=3: SSE=1,304.9  轮廓=0.1492  CH=65.0  DB=1.9072
  K=4: SSE=1,182.5  轮廓=0.1490  CH=60.1  DB=1.7066
  K=5: SSE=1,078.7  轮廓=0.1563  CH=58.0  DB=1.5879
  K=6: SSE=1,009.5  轮廓=0.1436  CH=54.4  DB=1.6610
  K=7: SSE=947.3  轮廓=0.1449  CH=52.1  DB=1.6296
  K=8: SSE=904.8  轮廓=0.1403  CH=49.0  DB=1.5530

轮廓系数最优 K = 2；肘部法 K = 5（两者不一致时以轮廓系数为主，并结合簇的业务含义）
[图片] 已保存：分类聚类_A_原始数据_选K指标.png（同名覆盖）

指标汇总表：
 K        SSE  轮廓系数      CH     DB
 2 1,493.3100    0.1590 67.9238 2.1826
 3 1,304.8990    0.1492 64.9651 1.9072
 4 1,182.5344    0.1490 60.1462 1.7066
 5 1,078.6616    0.1563 58.0078 1.5879
 6 1,009.5400    0.1436 54.3760 1.6610
 7   947.2977    0.1449 52.0879 1.6296
 8   904.8025    0.1403 49.0154 1.5530


### 7. K-Means 主聚类 + 簇画像 + 可视化

- `n_init=10` 多次初始化取 SSE 最小解；`random_state` 固定保证可复现；
- 用 **PCA 降到 2 维**画散点（只是可视化，聚类仍在高维标准化空间里做）；
- **簇画像表**：各簇在原始量纲下的数值均值 + 类别列的众数，这是给簇命名、写结论的依据；
- **轮廓图**：每簇样本轮廓值分布，能看出哪个簇分得最不干净。

In [14]:
# ============================================================
# 7. K-Means 聚类 + 簇画像 + PCA 散点 + 轮廓图
# ============================================================
def cluster_profile(df, labels, cfg, col_name="cluster"):
    """簇画像：数值列取均值、类别列取众数，全部用原始量纲，便于向评委解释。"""
    tmp = df.copy()
    tmp[col_name] = labels
    num = [c for c in df.columns if is_numeric_dtype(df[c])]
    cat = [c for c in df.columns if not is_numeric_dtype(df[c])
           and not is_datetime64_any_dtype(df[c]) and c not in (cfg.id_cols or ())]
    pieces = []
    if num:
        pieces.append(tmp.groupby(col_name)[num].mean().round(3))
    for c in cat:
        mode = tmp.groupby(col_name)[c].agg(
            lambda s: s.mode(dropna=True).iat[0] if len(s.mode(dropna=True)) else "未知")
        pieces.append(mode.rename(c).to_frame())
    prof = pd.concat(pieces, axis=1)
    prof.insert(0, "样本数", tmp.groupby(col_name).size())
    return prof


def plot_clusters(X, labels, centers, cfg, name="KMeans_PCA"):
    """PCA 2D 投影画聚类散点，并把簇中心也投影到同一坐标系。"""
    if not cfg.make_plots:
        return None                  # 不画图时 pca 对象为 None，不影响后续建模
    pca = PCA(n_components=2, random_state=cfg.random_state)
    Z = pca.fit_transform(X.to_numpy())
    C = pca.transform(np.asarray(centers))
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    for k in sorted(set(labels)):
        m = labels == k
        ax.scatter(Z[m, 0], Z[m, 1], s=22, alpha=0.75, label=f"簇 {k}（n={m.sum()}）")
    ax.scatter(C[:, 0], C[:, 1], marker="X", s=220, c="black", edgecolor="white",
               linewidth=1.2, label="簇中心")
    ax.set_xlabel(f"PC1（解释 {pca.explained_variance_ratio_[0]:.1%}）")
    ax.set_ylabel(f"PC2（解释 {pca.explained_variance_ratio_[1]:.1%}）")
    ax.set_title("K-Means 聚类结果（PCA 二维投影）")
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_fig(fig, name, cfg)
    return pca


def plot_silhouette(X, labels, cfg):
    """轮廓图：每个样本一条横线，虚线为总体平均轮廓系数。"""
    if not cfg.make_plots:
        return
    vals = silhouette_samples(X.to_numpy(), labels)
    avg = float(vals.mean())
    fig, ax = plt.subplots(figsize=(7, 5))
    y0 = 0
    for k in sorted(set(labels)):
        v = np.sort(vals[labels == k])
        ax.fill_betweenx(np.arange(y0, y0 + len(v)), 0, v, alpha=0.8, label=f"簇 {k}")
        ax.text(-0.06, y0 + len(v) / 2, str(k), fontsize=9)
        y0 += len(v)
    ax.axvline(avg, ls="--", color="red", label=f"平均 = {avg:.3f}")
    ax.set_xlabel("轮廓系数 s(i)"); ax.set_ylabel("样本（按簇排列）")
    ax.set_title("各簇轮廓系数分布")
    ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout()
    save_fig(fig, "轮廓图", cfg)


def run_kmeans(df, X, k, cfg):
    km = KMeans(n_clusters=k, n_init=10, random_state=cfg.random_state).fit(X)
    labels = km.labels_
    out = df.copy()
    out["cluster"] = labels                        # 簇编号写回原表，便于按簇筛选样本
    prof = cluster_profile(df, labels, cfg)
    pca = plot_clusters(X, labels, km.cluster_centers_, cfg)
    plot_silhouette(X, labels, cfg)
    print(f"K-Means 完成：K={k}，平均轮廓系数 = "
          f"{silhouette_score(X, labels):.4f}，各簇样本数 = "
          f"{dict(pd.Series(labels).value_counts().sort_index())}")
    print("\n簇画像（原始量纲）：")
    print(prof.to_string())
    return out, km, pca, prof


clu_df, KM, PCA_MODEL, PROFILE = run_kmeans(df, X, K_BEST, CONFIG)

[图片] 已保存：分类聚类_A_原始数据_KMeans_PCA.png（同名覆盖）
[图片] 已保存：分类聚类_A_原始数据_轮廓图.png（同名覆盖）
K-Means 完成：K=2，平均轮廓系数 = 0.1590，各簇样本数 = {0: np.int64(219), 1: np.int64(147)}

簇画像（原始量纲）：
         样本数    pm25  temperature  traffic_index  survey_score  household_income region fuel_type
cluster                                                                                             
0           219 49.6350      25.4430       100.5630       70.4470        5,799.2280   南部    天然气
1           147 52.0480      18.1780        97.6530       70.4610        5,617.9420   东部    天然气


### 8. 层次聚类（Ward）+ 谱系图 + 与 K-Means 的一致性

K-Means 需要预先指定 $K$，层次聚类则给出完整的聚类树，能从谱系图上直观看到
「样本是怎么一层层合并的」。用 **ARI（调整兰德指数）** 比较两种方法：
ARI 越接近 1，说明分群结果越稳定，论文的稳健性检验就有依据了。

In [16]:
# ============================================================
# 8. Ward 层次聚类 + 谱系图 + ARI 一致性检验
# ============================================================
def run_hierarchical(df, X, k, cfg, kmeans_labels=None):
    Xv = X.to_numpy()
    Z = linkage(Xv, method="ward")                 # Ward：合并使簇内方差增量最小的两簇
    labels = fcluster(Z, t=k, criterion="maxclust") - 1   # 转成 0 开始的编号，与 KMeans 对齐

    if cfg.make_plots:
        fig, ax = plt.subplots(figsize=(9, 4.2))
        dendrogram(Z, truncate_mode="lastp", p=30, show_leaf_counts=True, ax=ax,
                   color_threshold=None)
        ax.axhline(0.9 * Z[-1, 2], ls="--", color="#d95f5f", label="参考切分高度")
        ax.set_title("Ward 层次聚类谱系图（最后 30 次合并）")
        ax.set_xlabel("样本数 / 叶节点"); ax.set_ylabel("合并距离")
        ax.legend(fontsize=8)
        fig.tight_layout()
        save_fig(fig, "谱系图", cfg)

    if kmeans_labels is not None:
        ari = adjusted_rand_score(kmeans_labels, labels)
        print(f"层次聚类 vs K-Means 的 ARI = {ari:.4f}"
              f"（1 = 完全一致，0 = 随机水平；>0.6 说明分群结构较稳健）")
    sil = silhouette_score(Xv, labels) if len(set(labels)) > 1 else np.nan
    print(f"层次聚类轮廓系数 = {sil:.4f}；簇数 = {len(set(labels))}")
    prof = cluster_profile(df, labels, cfg)
    return labels, Z, prof


HIER_LABELS, LINK_MAT, HIER_PROFILE = run_hierarchical(df, X, K_BEST, CONFIG, KM.labels_)
print("\n层次聚类簇画像：")
print(HIER_PROFILE.to_string())

[图片] 已保存：分类聚类_A_原始数据_谱系图.png（同名覆盖）
层次聚类 vs K-Means 的 ARI = 0.6085（1 = 完全一致，0 = 随机水平；>0.6 说明分群结构较稳健）
层次聚类轮廓系数 = 0.1341；簇数 = 2

层次聚类簇画像：
         样本数    pm25  temperature  traffic_index  survey_score  household_income region fuel_type
cluster                                                                                             
0           133 52.2500      18.2750       100.5090       71.6870        5,672.5370   东部    天然气
1           233 49.6620      24.9400        98.7540       69.7530        5,737.9390   南部    天然气


### 9. DBSCAN 密度聚类：自动找离群样本

DBSCAN 不需要事先给定簇数，靠 $(eps,\ min\_samples)$ 决定簇；标签 `-1` 即噪声/离群点，
在美赛里常用来做**异常样本筛查**或**稳健性讨论**。
`eps` 由 k-距离曲线（第 `min_samples` 近邻的距离排序）的拐点自动确定，
拐点异常时用 `dbscan_eps_quantile` 分位数兜底。

In [18]:
# ============================================================
# 9. DBSCAN：k-距离曲线自动定 eps，输出噪声点清单
# ============================================================
def run_dbscan(df, X, cfg):
    Xv = X.to_numpy()
    ms = max(2, int(cfg.dbscan_min_samples))
    nn = NearestNeighbors(n_neighbors=ms).fit(Xv)
    dist, _ = nn.kneighbors(Xv)
    kdist = np.sort(dist[:, -1])                   # 每个点到第 ms 个近邻的距离
    idx = find_elbow(kdist)                        # 复用第 6 节的通用拐点函数
    eps = float(kdist[idx])
    # 拐点若落在曲线最左端（eps 过小会把几乎所有点判成噪声），改用高分位数兜底
    if not np.isfinite(eps) or eps < float(np.quantile(kdist, 0.50)):
        eps = float(np.quantile(kdist, cfg.dbscan_eps_quantile))
    print(f"min_samples = {ms}；拐点下标 {idx}/{len(kdist)} → eps = {eps:.4f}")

    if cfg.make_plots:
        fig, ax = plt.subplots(figsize=(6.8, 4))
        ax.plot(kdist, color="#3b7dd8")
        ax.axhline(eps, ls="--", color="#d95f5f", label=f"eps = {eps:.3f}")
        ax.set_xlabel("样本（按 k-距离升序）"); ax.set_ylabel(f"第 {ms} 近邻距离")
        ax.set_title("DBSCAN：k-距离曲线与 eps 选取")
        ax.legend(fontsize=8); fig.tight_layout()
        save_fig(fig, "DBSCAN_k距离", cfg)

    lab = DBSCAN(eps=eps, min_samples=ms).fit_predict(Xv)
    n_cluster = len(set(lab)) - (1 if -1 in lab else 0)
    n_noise = int((lab == -1).sum())
    print(f"DBSCAN：发现 {n_cluster} 个密度簇，噪声点 {n_noise} 个"
          f"（{n_noise / len(lab):.1%}）；噪声点即候选异常样本")

    out = df.copy()
    out["dbscan_cluster"] = lab
    noise = out[out["dbscan_cluster"] == -1]
    if len(noise):
        print("\n噪声样本前 5 条（可在论文里作为异常值讨论）：")
        print(noise.head().to_string())
    return out, lab, eps, noise


DBSCAN_DF, DB_LABELS, EPS, NOISE = run_dbscan(df, X, CONFIG)

min_samples = 5；拐点下标 307/366 → eps = 1.6240
[图片] 已保存：分类聚类_A_原始数据_DBSCAN_k距离.png（同名覆盖）
DBSCAN：发现 1 个密度簇，噪声点 13 个（3.6%）；噪声点即候选异常样本

噪声样本前 5 条（可在论文里作为异常值讨论）：
         date region    pm25  temperature  traffic_index fuel_type  survey_score  household_income  dbscan_cluster
18 2024-02-24   东部     NaN      23.1930       131.1886    生物质       90.2324               NaN              -1
32 2024-01-02   东部 69.9246      12.2709       109.4906    生物质       60.3289               NaN              -1
39 2024-02-06   南部 36.3547      24.3385        91.8229      电力       49.0352               NaN              -1
76 2024-01-09   东部 65.8587      15.0527        43.7888      电力       64.0043               NaN              -1
86 2024-03-22   东部 62.2474      27.6701        73.5156        煤       51.5165               NaN              -1


### 10. 有监督分类（有标签时才执行）

如果 `CONFIG.target` 指向一列标签：

- 标签是连续数值且取值很多 → 自动按三分位数分箱成 `低 / 中 / 高`；
- 4 个候选模型：**逻辑回归、随机森林、SVM(RBF)、KNN**，统一用分层 $k$ 折交叉验证比较
  Accuracy 与 Macro-F1（类别不均衡时 Macro 更诚实）；
- 取 CV 最优模型做留出集测试，给混淆矩阵、分类报告、ROC-AUC；
- 随机森林的特征重要度可用于「哪些指标最能区分等级」的解释。

若 `target=None`，`run_pipeline` 会整段跳过，只输出聚类结果。

In [20]:
# ============================================================
# 10. 分类：多模型 × 分层交叉验证 × 混淆矩阵 / AUC / 特征重要度
# ============================================================
def make_models(X, cfg):
    """候选分类器字典；数据规模很小，默认参数即可，换题目可在这里加网格搜索。"""
    return {
        "逻辑回归": LogisticRegression(max_iter=2000, random_state=cfg.random_state),
        "随机森林": RandomForestClassifier(n_estimators=300, random_state=cfg.random_state,
                                    n_jobs=-1),
        "SVM(RBF)": SVC(kernel="rbf", probability=True, random_state=cfg.random_state),
        "KNN": KNeighborsClassifier(n_neighbors=5),
    }


def prepare_target(df, cfg):
    """取标签列 → 必要时分箱 → LabelEncoder 编码，返回 (y整数编码, 编码器, 原始标签)。"""
    y_raw = df[cfg.target]
    if is_numeric_dtype(y_raw) and y_raw.nunique(dropna=True) > cfg.max_label_classes:
        y_raw = pd.qcut(y_raw, 3, labels=["低", "中", "高"], duplicates="drop")
        print(f"[提示] 标签 '{cfg.target}' 为连续值，已按三分位分箱为 低/中/高。")
    # 只用非缺失标签训练编码器，缺失位置记为 -1（后续由 mask 剔除），
    # 否则 'nan' 会被当成一个真实类别，导致 target_names 与类别数不匹配。
    valid = y_raw.notna()
    le = LabelEncoder()
    le.fit(y_raw.loc[valid].astype(str))
    y = np.full(len(df), -1, dtype=int)
    y[valid.to_numpy()] = le.transform(y_raw.loc[valid].astype(str))
    print(f"标签列 '{cfg.target}'：{len(le.classes_)} 类 → {list(le.classes_)}，"
          f"有效标签 {int(valid.sum())}/{len(df)} 条")
    return y, le, y_raw


def run_classification(df, X, cfg, extra_features=None):
    """完整分类流程；extra_features 可传入聚类标签的独热列，用于第 11 节的联动实验。"""
    y, le, y_raw = prepare_target(df, cfg)
    mask = df[cfg.target].notna().to_numpy()       # 标签缺失的样本不参与有监督训练
    Xm = X.loc[mask].copy()
    ym = y[mask]
    if extra_features is not None:
        Xm = pd.concat([Xm, extra_features.loc[mask]], axis=1)

    counts = pd.Series(ym).value_counts()
    n_splits = min(cfg.n_splits, int(counts.min()))
    if n_splits < 2:
        print("[警告] 某个类别样本数 < 2，跳过交叉验证。")
        return None
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=cfg.random_state)

    # ---- 10.1 交叉验证横向比较 ----
    rows = []
    for name, model in make_models(Xm, cfg).items():
        acc = cross_val_score(model, Xm, ym, cv=cv, scoring="accuracy")
        f1 = cross_val_score(model, Xm, ym, cv=cv, scoring="f1_macro")
        rows.append(dict(模型=name, CV_Accuracy均值=acc.mean(), CV_Accuracy标准差=acc.std(),
                         CV_F1均值=f1.mean(), CV_F1标准差=f1.std()))
    report = pd.DataFrame(rows).sort_values("CV_F1均值", ascending=False).reset_index(drop=True)
    print("\n===== 交叉验证模型对比（按 Macro-F1 排序）=====")
    print(report.to_string(index=False))
    best_name = report.loc[0, "模型"]

    # ---- 10.2 留出集测试 ----
    X_tr, X_te, y_tr, y_te = train_test_split(
        Xm, ym, test_size=cfg.test_size, stratify=ym, random_state=cfg.random_state)
    best = make_models(Xm, cfg)[best_name]
    best.fit(X_tr, y_tr)
    pred = best.predict(X_te)
    metrics = dict(模型=best_name,
                   Accuracy=accuracy_score(y_te, pred),
                   Macro_Precision=precision_score(y_te, pred, average="macro", zero_division=0),
                   Macro_Recall=recall_score(y_te, pred, average="macro", zero_division=0),
                   Macro_F1=f1_score(y_te, pred, average="macro", zero_division=0))
    try:
        proba = best.predict_proba(X_te)
        metrics["ROC_AUC"] = (roc_auc_score(y_te, proba[:, 1]) if len(le.classes_) == 2
                              else roc_auc_score(y_te, proba, multi_class="ovr"))
    except Exception as e:
        metrics["ROC_AUC"] = np.nan
        print("[提示] AUC 计算跳过：", type(e).__name__)
    print("\n===== 留出集表现 =====")
    print(pd.DataFrame([metrics]).to_string(index=False))
    # 显式指定 labels，保证某个类别碰巧没出现在测试集时长度也不会错位
    all_labels = list(range(len(le.classes_)))
    print("\n分类报告：")
    print(classification_report(y_te, pred, labels=all_labels,
                                target_names=le.classes_, zero_division=0))

    # ---- 10.3 混淆矩阵 ----
    if cfg.make_plots:
        fig, ax = plt.subplots(figsize=(5.5, 4.5))
        ConfusionMatrixDisplay.from_predictions(y_te, pred, labels=all_labels,
                                                display_labels=le.classes_,
                                                cmap="Blues", ax=ax, colorbar=False)
        ax.set_title(f"混淆矩阵（{best_name}）")
        fig.tight_layout()
        save_fig(fig, "混淆矩阵", cfg)

    # ---- 10.4 特征重要度（用随机森林统一给出，不依赖最优模型是不是树）----
    rf = RandomForestClassifier(n_estimators=300, random_state=cfg.random_state,
                                n_jobs=-1).fit(X_tr, y_tr)
    imp = (pd.Series(rf.feature_importances_, index=Xm.columns)
           .sort_values(ascending=False).head(12).iloc[::-1])
    if cfg.make_plots:               # 重要度数值照算（要写进 Excel），只是不画图
        fig, ax = plt.subplots(figsize=(7, max(3, 0.35 * len(imp))))
        ax.barh(imp.index, imp.values, color="#4a9d6a")
        ax.set_xlabel("随机森林特征重要度"); ax.set_title("哪些指标最能区分类别")
        fig.tight_layout()
        save_fig(fig, "特征重要度", cfg)

    return dict(report=report, metrics=metrics, model=best, label_encoder=le,
                best_name=best_name, X=Xm, y=ym, feature_importance=imp,
                classification_report=classification_report(
                    y_te, pred, labels=all_labels, target_names=le.classes_,
                    zero_division=0))


# 若配置了标签列，就执行一次分类；target=None 时这里什么也不做
CLS = run_classification(df, X, CONFIG) if CONFIG.target is not None else None
if CLS is None:
    print("CONFIG.target = None → 已跳过分类环节，本次只做聚类。")

标签列 'region'：4 类 → ['东部', '北部', '南部', '西部']，有效标签 366/366 条

===== 交叉验证模型对比（按 Macro-F1 排序）=====
    模型  CV_Accuracy均值  CV_Accuracy标准差  CV_F1均值  CV_F1标准差
逻辑回归           0.6722             0.0629     0.6704       0.0580
随机森林           0.6528             0.0692     0.6446       0.0686
SVM(RBF)           0.6203             0.0512     0.6172       0.0550
     KNN           0.5435             0.0629     0.5498       0.0608

===== 留出集表现 =====
    模型  Accuracy  Macro_Precision  Macro_Recall  Macro_F1  ROC_AUC
逻辑回归    0.6739           0.6925        0.6739    0.6808   0.8855

分类报告：
              precision    recall  f1-score   support

          东部       0.95      0.78      0.86        23
          北部       0.52      0.57      0.54        23
          南部       0.52      0.57      0.54        23
          西部       0.78      0.78      0.78        23

    accuracy                           0.67        92
   macro avg       0.69      0.67      0.68        92
weighted avg       0.69      0.67      0.6

### 11. 分类 × 聚类联动：簇标签能不能提升判别？

美赛里常见的加分做法：**先无监督分群，再把「属于哪个群」作为新特征喂给分类器**。
如果加入簇标签后 Macro-F1 明显提升，说明数据里确实存在「先分群、再分类」的层次结构，
可以据此写出「分区/分型治理」之类的结论。同时给出 簇 × 真实类别 的交叉表，看簇与类别的对应关系。

In [22]:
# ============================================================
# 11. 联动实验：聚类簇标签作为额外特征，比较分类效果
# ============================================================
def run_linkage(df, X, labels, cfg):
    """把 KMeans 簇编号做成独热特征，比较加入前后的交叉验证 Macro-F1。"""
    if cfg.target is None or df[cfg.target].isna().all():
        print("没有标签列 → 跳过联动实验（该实验需要有监督标签）。")
        return None
    onehot = pd.get_dummies(pd.Series(labels, index=X.index, name="cluster"),
                            prefix="cluster").astype(float)
    y, le, _ = prepare_target(df, cfg)
    mask = df[cfg.target].notna().to_numpy()
    y = y[mask]
    cv = StratifiedKFold(n_splits=min(cfg.n_splits, int(pd.Series(y).value_counts().min())),
                         shuffle=True, random_state=cfg.random_state)

    rows = []
    for tag, X_use in [("原始特征", X.loc[mask]),
                       ("原始特征 + 簇标签", pd.concat([X.loc[mask], onehot.loc[mask]], axis=1))]:
        model = RandomForestClassifier(n_estimators=300, random_state=cfg.random_state,
                                       n_jobs=-1)
        f1 = cross_val_score(model, X_use, y, cv=cv, scoring="f1_macro")
        acc = cross_val_score(model, X_use, y, cv=cv, scoring="accuracy")
        rows.append(dict(特征方案=tag, CV_F1均值=f1.mean(), CV_F1标准差=f1.std(),
                         CV_Accuracy均值=acc.mean()))
    cmp = pd.DataFrame(rows)
    delta = float(cmp.loc[1, "CV_F1均值"] - cmp.loc[0, "CV_F1均值"])
    print("===== 加入簇标签前后的随机森林表现 =====")
    print(cmp.to_string(index=False))
    print(f"Macro-F1 变化 = {delta:+.4f} "
          f"（{'簇结构对判别有帮助' if delta > 0.01 else '提升不明显，可直接用原始特征'}）")

    # 簇 × 真实类别交叉表：看每个簇主要对应哪一类
    cross = pd.crosstab(pd.Series(labels, index=df.index, name="聚类簇"),
                        df[cfg.target].astype(str), normalize="index").round(3)
    print("\n簇 × 类别 构成比（按行归一化）：")
    print(cross.to_string())
    return dict(comparison=cmp, delta=delta, crosstab=cross, onehot=onehot)


LINK = run_linkage(df, X, KM.labels_, CONFIG)

标签列 'region'：4 类 → ['东部', '北部', '南部', '西部']，有效标签 366/366 条
===== 加入簇标签前后的随机森林表现 =====
         特征方案  CV_F1均值  CV_F1标准差  CV_Accuracy均值
         原始特征     0.6446       0.0686           0.6528
原始特征 + 簇标签     0.6315       0.0777           0.6392
Macro-F1 变化 = -0.0131 （提升不明显，可直接用原始特征）

簇 × 类别 构成比（按行归一化）：
region   东部   北部   南部   西部
聚类簇                            
0      0.2370 0.2470 0.2650 0.2510
1      0.2720 0.2450 0.2380 0.2450


### 12. 结果导出 + 新数据预测（真正「可复用」的两步）

- `export_results`：把聚类明细、簇画像、选 K 指标、分类报告、噪声点清单写成一个多 Sheet 的 Excel，
  同时另存 CSV；图片已在前面各节单独保存；
- `predict_new`：拿任意一份**结构相同的新数据**，用训练时保存的预处理器 + 模型直接出簇号和类别预测，
  不需要重新拟合，也不会发生数据泄漏；
- 如需把模型带到别的 Notebook，可用 `joblib.dump(bundle, path)` 保存整个 bundle。

In [24]:
# ============================================================
# 12. 导出结果与新数据推理
# ============================================================
def clean_outputs(tag=None):
    """清理某个 tag 遗留的全部产物（只匹配本模板的 分类聚类_{tag}_* 前缀，不碰别的文件）。

    典型用途：换 tag 后旧的一组图片/表格没用了，调用 clean_outputs("raw_demo") 即可。
    """
    tag = CONFIG.tag if tag is None else tag
    files = sorted(OUT_DIR.glob(f"分类聚类_{tag}_*"))
    for f in files:
        f.unlink()
    print(f"已清理 tag='{tag}' 的 {len(files)} 个文件：",
          [f.name for f in files] if files else "无")
    return files


def export_results(bundle, cfg):
    """把全部关键结果写进 输出数据/：CSV 明细 + 多 Sheet Excel 汇总。"""
    tag = cfg.tag
    clu = bundle["clu_df"].copy()
    if bundle.get("dbscan_df") is not None:
        clu["dbscan_cluster"] = bundle["dbscan_df"]["dbscan_cluster"].to_numpy()
    clu["hier_cluster"] = bundle.get("hier_labels")
    csv_path = OUT_DIR / f"分类聚类_{tag}_聚类明细.csv"
    clu.to_csv(csv_path, index=False, encoding="utf-8-sig")

    sheets = {
        "数据体检": bundle["audit"],
        "选K指标": bundle["k_table"],
        "簇画像_KMeans": bundle["profile"].reset_index(),
    }
    if bundle.get("hier_profile") is not None:
        sheets["簇画像_层次"] = bundle["hier_profile"].reset_index()
    if bundle.get("noise") is not None and len(bundle["noise"]):
        sheets["DBSCAN噪声点"] = bundle["noise"]
    if bundle.get("cls") is not None:
        sheets["分类_模型对比"] = bundle["cls"]["report"]
        sheets["分类_留出集指标"] = pd.DataFrame([bundle["cls"]["metrics"]])
        sheets["分类_特征重要度"] = bundle["cls"]["feature_importance"].rename("重要度").reset_index()
    if bundle.get("link") is not None:
        sheets["分类聚类联动"] = bundle["link"]["comparison"]

    xlsx_path = OUT_DIR / f"分类聚类_{tag}_结果汇总.xlsx"
    try:
        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as w:
            for name, table in sheets.items():
                table.to_excel(w, sheet_name=name[:31], index=False)   # Excel sheet 名上限 31 字符
        print(f"[Excel] {xlsx_path.name}（{len(sheets)} 个工作表）")
    except ImportError:
        print("[提示] 未安装 openpyxl，跳过 Excel；CSV 已正常导出。 "
              "安装命令：pip install openpyxl")
    print(f"[CSV]   {csv_path.name}")
    return csv_path, xlsx_path


def predict_new(bundle, df_new):
    """对新数据做推理：复用同一个预处理器 / K-Means / 分类器，输出簇号与类别预测。

    要求 df_new 的列名与训练数据一致；缺失值会按训练集的参数自动填补。
    """
    cfg = bundle["cfg"]
    X_new = bundle["preprocessor"].transform(df_new)
    out = df_new.copy()
    out["cluster_pred"] = bundle["kmeans"].predict(X_new)
    if bundle.get("cls") is not None:
        cls = bundle["cls"]
        y_pred = cls["model"].predict(X_new)
        out["class_pred"] = cls["label_encoder"].inverse_transform(y_pred)
    print(f"新数据 {len(out)} 行完成推理；预测簇分布："
          f"{dict(pd.Series(out['cluster_pred']).value_counts().sort_index())}")
    return out


# 演示：把当前数据的最后 5 行当作「新来的一批样本」，检查复用链路是否通畅
BUNDLE_DEMO = dict(cfg=CONFIG, preprocessor=pre, kmeans=KM, clu_df=clu_df,
                   audit=audit, k_table=k_table, profile=PROFILE,
                   dbscan_df=DBSCAN_DF, hier_labels=HIER_LABELS,
                   hier_profile=HIER_PROFILE, noise=NOISE, cls=CLS, link=LINK)
new_pred = predict_new(BUNDLE_DEMO, df.tail(5))
print(new_pred.to_string())

新数据 5 行完成推理；预测簇分布：{0: np.int64(3), 1: np.int64(2)}
          date region    pm25  temperature  traffic_index fuel_type  survey_score  household_income  cluster_pred class_pred
361 2024-02-23   北部 36.4961      24.3966       113.3143        煤       72.3685               NaN             0       西部
362 2024-01-13   南部 42.1176      16.3983        87.0171      电力       60.7993               NaN             1       南部
363 2024-01-28   西部 36.3531      19.4858            NaN    天然气       64.6493               NaN             1       西部
364 2024-02-17   东部 61.4731      26.3702        89.1718    天然气       73.7182               NaN             0       东部
365 2024-03-23   西部 40.6825      28.2689       114.7221        煤       68.5508               NaN             0       西部


### 13. 一键入口：一行换数据 `run_on`（最推荐）

`run_on` 是 `run_pipeline` 的傻瓜包装：**给文件名、标签列、tag，其余全不用管**。
它会自动拼路径、检查文件、创建配置、跑完整个流程，并把所有结果装进 `bundle` 返回。

```python
# 换题目时，只改这一行的三个值就够了
bundle = run_on("新附件.xlsx", target="等级", tag="problem_e")

# 没有标签列 → 只聚类
bundle = run_on("新附件.xlsx", target=None, tag="problem_e")

# 顺手调参数（任意 Config 字段都能当关键字参数传）
bundle = run_on("新附件.xlsx", target="等级", tag="problem_e",
                k_max=10, make_plots=False, sheet_name="Sheet1")

# 新来一批数据，复用训练好的模型直接推理
result = predict_new(bundle, 新数据_df)
```

等价的完整写法（想逐项调参、做灵敏度分析时用）：

```python
cfg = Config(data_path=DATA_DIR / "新附件.xlsx",
             target="等级", features=None, tag="problem_e", k_max=10)
bundle = run_pipeline(cfg)
```

`bundle` 里装着所有中间对象：清洗后的表 `df`、特征矩阵 `X`、预处理器 `preprocessor`、
聚类模型 `kmeans`、簇画像 `profile`、分类器 `cls["model"]`，方便继续画图或二次分析。

In [26]:
# ============================================================
# 13. 主流程：一个函数跑完全部环节
# ============================================================
def run_pipeline(cfg, verbose=True):
    """读取 → 体检 → 预处理 → 选 K → 三种聚类 → 分类 → 联动 → 导出。

    返回 bundle：所有对象都在里面，可继续 predict_new / joblib.dump / 二次分析。
    """
    print("=" * 70)
    print(f"开始建模：{cfg.tag}  数据={cfg.data_path}  标签={cfg.target}")
    print("=" * 70)

    # 1) 读取 + 体检
    d, meta = load_table(cfg)
    # 标签列名写错是最常见的报错，这里提前给出可读的提示
    if cfg.target is not None and cfg.target not in d.columns:
        raise KeyError(f"标签列 '{cfg.target}' 不存在。当前数据的列名：{list(d.columns)}")
    aud = data_audit(d, cfg)

    # 2) 预处理（fit 只在当前数据上做一次）
    p = TabularPreprocessor(cfg).fit(d)
    Xm = p.transform(d)
    print(f"\n特征矩阵：{Xm.shape}；被剔除列：{p.dropped_cols_ or '无'}")

    # 3) 选 K + K-Means
    kt, kb, ke = evaluate_k(Xm, cfg)
    plot_k_diagnostics(kt, kb, ke, cfg)
    clu, km, pca, prof = run_kmeans(d, Xm, kb, cfg)

    # 4) 层次聚类 + DBSCAN（互相印证 + 找离群点）
    hl, lk, hp = run_hierarchical(d, Xm, kb, cfg, km.labels_)
    db, dbl, eps, noise = run_dbscan(d, Xm, cfg)

    # 5) 有监督分类 + 分类聚类联动（无标签时自动跳过）
    cls = run_classification(d, Xm, cfg) if cfg.target is not None else None
    link = run_linkage(d, Xm, km.labels_, cfg) if cfg.target is not None else None
    if cls is None:
        print("\n[说明] target=None：已跳过分类与联动，仅输出聚类结果。")

    # 6) 汇总与导出
    bundle = dict(cfg=cfg, meta=meta, df=d, audit=aud, preprocessor=p, X=Xm,
                  k_table=kt, k_best=kb, k_elbow=ke, clu_df=clu, kmeans=km,
                  pca=pca, profile=prof, hier_labels=hl, link_mat=lk,
                  hier_profile=hp, dbscan_df=db, dbscan_labels=dbl, eps=eps,
                  noise=noise, cls=cls, link=link)
    export_results(bundle, cfg)
    print("=" * 70)
    print(f"完成：{cfg.tag}  最优 K={kb}（肘部法 K={ke}）；"
          f"特征 {Xm.shape[1]} 个；结果见 {OUT_DIR}")
    print("=" * 70)
    return bundle


def run_on(data_path, target=None, tag="my_data", **overrides):
    """一行换数据：给文件名 → 自动定位 → 建配置 → 跑完整流程 → 返回 bundle。

    data_path : 只写文件名（自动去 `导入数据/` 找），也可给完整绝对路径；
                写成 "导入数据/xxx.csv" 同样能识别，不会重复拼目录。
    target    : 标签列名；None = 没有标签，只做聚类（自动跳过分类与联动）。
    tag       : 本次输出文件前缀；换数据记得换名，否则会覆盖上一份结果。
    overrides : 任意 Config 字段，如 k_max=10 / make_plots=False / sheet_name="Sheet1"。

    示例::

        bundle = run_on("问题E数据.csv", target="风险等级", tag="problem_e")
        result = predict_new(bundle, 新数据_df)
    """
    p = Path(data_path)
    if not p.is_absolute():
        # 防呆：用户可能写 "xxx.csv"，也可能写 "导入数据/xxx.csv"，两种都正确定位
        p = (ROOT / p) if (p.parts and p.parts[0] == DATA_DIR.name) else (DATA_DIR / p)
    if not p.exists():
        raise FileNotFoundError(
            f"找不到数据文件：{p}\n请把文件放进 {DATA_DIR}，或传入完整绝对路径。")

    # 拼错参数名时给出可用字段清单，避免静默出错
    valid = {f.name for f in fields(Config)}
    unknown = set(overrides) - valid
    if unknown:
        raise ValueError(f"未知配置项 {sorted(unknown)}；可用字段：{sorted(valid)}")

    cfg = Config(data_path=p, target=target, tag=tag, **overrides)
    return run_pipeline(cfg)

### 14. 三个可直接照抄的运行示例

- **示例 A**：原始脏数据（有 BOM、哨兵值、文本数字、高缺失列、重复行），聚类 + `region` 分类；
- **示例 B**：换成已经清洗好的数据，且 `target=None` → **只做聚类**，验证「无标签也能跑」；
- **示例 C**：用示例 A 训练好的 bundle 对新数据推理（已在第 12 节演示，这里给出标准写法）。

想改成自己的题目，复制示例 A 的那两行、换掉 `data_path` / `target` / `tag` 即可。

> **嫌输出文件多？** 把第 2 节的 `RUN_EXAMPLES = False`，这三格就都不会执行，
> 一次运行只产生 `CONFIG.tag` 对应的一组结果；再配合 `make_plots=False` 可以一张图都不生成。
> 旧 tag 留下的重复文件用 `clean_outputs("raw_demo")` 一键清掉。

In [28]:
# ============================================================
# 14. 示例 A：原始脏数据，全流程（聚类 + 分类 + 联动 + 导出）
# ============================================================
if RUN_EXAMPLES:
    cfg_a = Config(data_path=DATA_DIR / "mcm_final_raw.csv",
                   target="region",       # 演示用标签；换成自己的标签列名
                   tag="A_原始数据")
    bundle_a = run_pipeline(cfg_a)
else:
    print("已跳过示例 A（RUN_EXAMPLES=False），正式复用时不必跑这些演示单元。")

开始建模：A_原始数据  数据=D:\git项目\导入数据\mcm_final_raw.csv  标签=region
读取成功 : mcm_final_raw.csv  （编码 utf-8-sig）
数据规模 : 381 行 → 366 行，8 列；删除重复行 15 行
文本转数值列 : ['pm25']；日期列：['date']
哨兵值转缺失 : {'traffic_index': 6}
===== 数据体检表 =====
            列名           类型  缺失数  缺失率  唯一值     最小值     中位数     最大值  IQR异常点               建议
            date datetime64[us]       0  0.0000      90                                           0 日期/标识列,不建模
          region            str       0  0.0000       4                                           0           独热编码
            pm25        float64      18  0.0492     344    18.6900    50.4260    93.2070          1               保留
     temperature        float64       5  0.0137     356    11.8600    23.2560    30.6550          0               保留
   traffic_index        float64      15  0.0410     351    43.7890    98.9720   150.0030          4               保留
       fuel_type            str       8  0.0219       4                                           0           独热编码
 

In [29]:
# ============================================================
# 15. 示例 B：清洗后数据 + 无标签（只聚类，分类环节自动跳过）
#     改 3 个参数即可复用到任何新附件
# ============================================================
clean_path = OUT_DIR / "final_cleaned.csv"
if RUN_EXAMPLES and clean_path.exists():
    cfg_b = Config(data_path=clean_path,
                   target=None,            # 无标签 → 自动跳过第 10、11 节
                   tag="B_清洗数据",
                   k_max=6,               # 样本少一点，少试几个 K
                   outlier="winsor_3sigma",  # 演示换一种缩尾方式
                   scale="minmax")           # 演示换一种标准化方式
    bundle_b = run_pipeline(cfg_b)
    print("示例 B 只做了聚类；最优 K =", bundle_b["k_best"])
elif not RUN_EXAMPLES:
    print("已跳过示例 B（RUN_EXAMPLES=False）。")
else:
    print("未找到 final_cleaned.csv，跳过示例 B（不影响模板使用）。")

开始建模：B_清洗数据  数据=D:\git项目\输出数据\final_cleaned.csv  标签=None
读取成功 : final_cleaned.csv  （编码 utf-8-sig）
数据规模 : 360 行 → 360 行，8 列；删除重复行 0 行
文本转数值列 : 无；日期列：['date']
哨兵值转缺失 : 无
===== 数据体检表 =====
                列名           类型  缺失数  缺失率  唯一值  最小值  中位数   最大值  IQR异常点               建议
                date datetime64[us]       0  0.0000      90                                   0 日期/标识列,不建模
              region            str       0  0.0000       4                                   0           独热编码
                pm25        float64       0  0.0000     345 18.6900 51.4780  93.2070          0               保留
         temperature        float64       0  0.0000     360 11.8600 23.1970  30.6550          0               保留
       traffic_index        float64       0  0.0000     351 43.7890 98.9720 150.0030          6               保留
           fuel_type            str       0  0.0000       4                                   0           独热编码
        survey_score        float64      18  0.0500     34

In [30]:
# ============================================================
# 16. 示例 C：复用训练好的模型处理新数据 / 保存模型
# ============================================================
# 标准写法（把 df_new 换成新附件读入的 DataFrame 即可）：
#     df_new, _ = load_table(Config(data_path=DATA_DIR / "新数据.csv", target=None))
#     result = predict_new(bundle_a, df_new)
#     result.to_csv(OUT_DIR / "新数据_分类聚类结果.csv", index=False, encoding="utf-8-sig")

# 若要把整套模型带到别的 Notebook / 第二天继续用：
# import joblib
# joblib.dump(bundle_a, OUT_DIR / "分类聚类_bundle.joblib")
# bundle = joblib.load(OUT_DIR / "分类聚类_bundle.joblib")

if RUN_EXAMPLES and "bundle_a" in globals():
    result_c = predict_new(bundle_a, bundle_a["df"].head(8))
    print("\n前 8 行的簇预测与类别预测：")
    print(result_c[["cluster_pred"] + (["class_pred"] if "class_pred" in result_c else [])].to_string())
    print("\n全部完成 ✅  结果文件都在：", OUT_DIR)
else:
    print("已跳过示例 C（RUN_EXAMPLES=False）。")

新数据 8 行完成推理；预测簇分布：{0: np.int64(6), 1: np.int64(2)}

前 8 行的簇预测与类别预测：
   cluster_pred class_pred
0             0       西部
1             0       北部
2             0       南部
3             1       北部
4             1       北部
5             0       东部
6             0       东部
7             0       东部

全部完成 ✅  结果文件都在： D:\git项目\输出数据
